# Data Wrangling: Fannie Mae Mortgage Delinquency Risk

For this project, I am using Fannie Mae loan performance data to build a loan-month dataset for delinquency prediction. Each row represents one loan during one reporting month.

This revised wrangling notebook is built to handle full-size quarterly files on a desktop machine:

- **Per-quarter processing.** Each raw file is processed and labeled on its own, then the per-quarter panels are concatenated. Fannie Mae performance files are organized by acquisition quarter, so a loan's full timeline lives in a single file and the target can be built per file. The notebook verifies this assumption after combining.
- **Chunked C-engine reads.** Each file is streamed in chunks with pandas' default C parser, and every chunk is cleaned, typed, and reduced before the next chunk is read, so the raw all-string data never sits in memory at once.
- **Vectorized parsing.** Dates, numerics, and delinquency codes are converted with vectorized pandas operations instead of per-value Python functions, and payment-history features are computed once per unique history string instead of once per row.
- **Loan sampling knob.** `LOAN_SAMPLE_FRACTION` keeps a deterministic, hash-based share of loans. Sampling is done by loan, not by row, so every kept loan keeps its complete timeline and the future-looking target stays valid. Set it to `None` on a machine with more memory to process every loan.

The main output is a tagged processed panel:

`data/processed/loan_month_panel_<DATASET_TAG>.parquet`

The target is `future_serious_dq_6m`, which flags whether a loan becomes 90 or more days delinquent within the next 6 observed reporting months.

## 1. Setup

This section imports the required libraries and defines the project folders, performance knobs, and run tag.

In [1]:
import gc
import glob
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from pandas.api.types import is_string_dtype

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [2]:
def find_project_root(start_path=None):
    start_path = Path.cwd() if start_path is None else Path(start_path)
    candidates = [start_path] + list(start_path.parents)
    for candidate in candidates:
        if (candidate / "data").exists() or candidate.name.lower() in {"notebooks", "notebook"}:
            if candidate.name.lower() in {"notebooks", "notebook"}:
                return candidate.parent
            return candidate
    return start_path.parent if start_path.name.lower() in {"notebooks", "notebook"} else start_path


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

for path in [RAW_DIR, INTERIM_DIR, PROCESSED_DIR, REPORTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

RAW_FILE_TOKENS = ["2021Q1", "2021Q2"]

HORIZON_MONTHS = 6
SERIOUS_DQ_THRESHOLD = 3
TARGET_COL = f"future_serious_dq_{HORIZON_MONTHS}m"

# Performance knobs for large quarterly files.
ROW_LIMIT = None              # Max raw rows per file. Set small (e.g. 500_000) for a quick smoke test.
MAX_FILES = None              # Max number of raw files to process.
CHUNKSIZE = 500_000           # Raw rows per read chunk. Lower if memory gets tight, raise with more RAM.
LOAN_SAMPLE_FRACTION = 0.10   # Share of loans to keep; each kept loan keeps its full timeline.
                              # Set to None or 1.0 to keep every loan (needs much more RAM).

DATASET_BASE_TAG = "2021q1_q2"

if LOAN_SAMPLE_FRACTION is None or LOAN_SAMPLE_FRACTION >= 1:
    DATASET_TAG = f"{DATASET_BASE_TAG}_full"
else:
    DATASET_TAG = f"{DATASET_BASE_TAG}_sample{int(round(LOAN_SAMPLE_FRACTION * 100))}pct"

PANEL_OUTPUT_PARQUET = PROCESSED_DIR / f"loan_month_panel_{DATASET_TAG}.parquet"
PANEL_OUTPUT_CSV = PROCESSED_DIR / f"loan_month_panel_{DATASET_TAG}.csv"
DATA_DICTIONARY_OUTPUT = PROCESSED_DIR / f"data_dictionary_{DATASET_TAG}.csv"
TARGET_SUMMARY_OUTPUT = PROCESSED_DIR / f"target_summary_{DATASET_TAG}.csv"
LOAN_TIMELINE_OUTPUT = PROCESSED_DIR / f"loan_timeline_gap_check_{DATASET_TAG}.csv"
RAW_PROFILE_OUTPUT = PROCESSED_DIR / f"raw_profile_{DATASET_TAG}.csv"

print("Dataset tag:", DATASET_TAG)

Dataset tag: 2021q1_q2_sample10pct


## 2. Selected Fannie Mae fields

The full Fannie Mae performance files contain more fields than this project needs. The list below keeps the columns needed for the loan-month panel, the 6-month delinquency target, the behavior features, leakage review, and the downstream preprocessing notebook.

In [3]:
KNOWN_FANNIE_FIELDS = [
    "reference_pool_id",
    "loan_identifier",
    "monthly_reporting_period",
    "channel",
    "seller_name",
    "servicer_name",
    "master_servicer",
    "original_interest_rate",
    "current_interest_rate",
    "original_upb",
    "upb_at_issuance",
    "current_actual_upb",
    "original_loan_term",
    "origination_date",
    "first_payment_date",
    "loan_age",
    "remaining_months_to_legal_maturity",
    "remaining_months_to_maturity",
    "maturity_date",
    "original_ltv",
    "original_cltv",
    "number_of_borrowers",
    "dti",
    "borrower_credit_score_at_origination",
    "co_borrower_credit_score_at_origination",
    "first_time_home_buyer_indicator",
    "loan_purpose",
    "property_type",
    "number_of_units",
    "occupancy_status",
    "property_state",
    "msa",
    "zip_code_short",
    "mortgage_insurance_percentage",
    "amortization_type",
    "prepayment_penalty_indicator",
    "interest_only_loan_indicator",
    "interest_only_first_principal_and_interest_payment_date",
    "months_to_amortization",
    "current_loan_delinquency_status",
    "loan_payment_history",
    "modification_flag",
    "mortgage_insurance_cancellation_indicator",
    "zero_balance_code",
    "zero_balance_effective_date",
    "upb_at_time_of_removal",
    "repurchase_date",
    "scheduled_principal_current",
    "total_principal_current",
    "unscheduled_principal_current",
    "last_paid_installment_date",
    "foreclosure_date",
    "disposition_date",
    "foreclosure_costs",
    "property_preservation_and_repair_costs",
    "asset_recovery_costs"
]

SELECTED_RAW_FIELDS = [
    "reference_pool_id",
    "loan_identifier",
    "monthly_reporting_period",
    "channel",
    "seller_name",
    "servicer_name",
    "original_interest_rate",
    "current_interest_rate",
    "original_upb",
    "upb_at_issuance",
    "current_actual_upb",
    "original_loan_term",
    "origination_date",
    "first_payment_date",
    "loan_age",
    "remaining_months_to_legal_maturity",
    "remaining_months_to_maturity",
    "maturity_date",
    "original_ltv",
    "original_cltv",
    "number_of_borrowers",
    "dti",
    "borrower_credit_score_at_origination",
    "co_borrower_credit_score_at_origination",
    "first_time_home_buyer_indicator",
    "loan_purpose",
    "property_type",
    "number_of_units",
    "occupancy_status",
    "property_state",
    "msa",
    "zip_code_short",
    "mortgage_insurance_percentage",
    "amortization_type",
    "prepayment_penalty_indicator",
    "interest_only_loan_indicator",
    "current_loan_delinquency_status",
    "loan_payment_history",
    "modification_flag",
    "zero_balance_code",
    "zero_balance_effective_date",
    "upb_at_time_of_removal",
    "repurchase_date",
    "scheduled_principal_current",
    "total_principal_current",
    "unscheduled_principal_current",
    "last_paid_installment_date",
    "foreclosure_date",
    "disposition_date"
]

COLUMN_RENAME_MAP = {
    "original_loan_to_value_ratio_ltv": "original_ltv",
    "original_combined_loan_to_value_ratio_cltv": "original_cltv",
    "debt_to_income_dti": "dti",
    "zip3": "zip_code_short",
    "current_delinquency_status": "current_loan_delinquency_status",
    "reporting_period": "monthly_reporting_period",
    "loan_id": "loan_identifier"
}

## 3. Helper functions

These functions detect the raw file layout, read only the selected fields, standardize column names, convert dates and numbers, and build the target.

In [4]:
def clean_column_names(columns):
    cleaned = (
        pd.Index(columns)
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    return cleaned.tolist()


def standardize_column_name(column):
    cleaned = clean_column_names([column])[0]
    return COLUMN_RENAME_MAP.get(cleaned, cleaned)


def detect_delimiter(file_path, sample_rows=5):
    candidates = ["|", ",", "	"]
    scores = []

    for delimiter in candidates:
        try:
            preview = pd.read_csv(
                file_path,
                sep=delimiter,
                header=None,
                nrows=sample_rows,
                dtype="string",
                engine="python"
            )
            scores.append((preview.dropna(axis=1, how="all").shape[1], delimiter))
        except Exception:
            scores.append((0, delimiter))

    return max(scores, key=lambda item: item[0])[1]


def looks_like_header(first_line, delimiter):
    parts = first_line.strip().split(delimiter)
    standardized_parts = [standardize_column_name(part) for part in parts]
    known_matches = sum(part in KNOWN_FANNIE_FIELDS or part in SELECTED_RAW_FIELDS for part in standardized_parts)
    return known_matches >= 2


def build_column_names(n_cols):
    names = []
    for i in range(n_cols):
        if i < len(KNOWN_FANNIE_FIELDS):
            names.append(KNOWN_FANNIE_FIELDS[i])
        else:
            names.append(f"field_{i+1:03d}")
    return names

In [5]:
def parse_mmyyyy_series(series):
    s = series.astype("string").str.strip()
    s = s.str.replace(r"\.0$", "", regex=True)

    mmyyyy_mask = s.str.fullmatch(r"\d{5,6}", na=False)
    s = s.where(~mmyyyy_mask, s.str.zfill(6))

    parsed = pd.to_datetime(s, format="%m%Y", errors="coerce")

    fallback_mask = s.notna() & parsed.isna()

    if fallback_mask.any():
        parsed.loc[fallback_mask] = pd.to_datetime(s[fallback_mask], errors="coerce")

    return parsed


def delinquency_to_numeric_series(series):
    # Non-numeric codes such as "XX" coerce to NaN.
    s = series.astype("string").str.strip()
    return pd.to_numeric(s, errors="coerce").astype("float32")


def parse_payment_history_codes(value):
    if pd.isna(value):
        return []

    text = str(value).strip().upper().replace(" ", "")

    if text == "":
        return []

    if len(text) % 2 != 0:
        text = text[1:]

    return [text[i:i+2] for i in range(0, len(text), 2)]


def payment_history_count_dq(value, threshold=1, lookback=12):
    codes = parse_payment_history_codes(value)
    recent_codes = codes[-lookback:]
    numeric_codes = [int(code) for code in recent_codes if code.isdigit()]
    return sum(code >= threshold for code in numeric_codes)


def payment_history_max_dq(value, lookback=12):
    codes = parse_payment_history_codes(value)
    recent_codes = codes[-lookback:]
    numeric_codes = [int(code) for code in recent_codes if code.isdigit()]

    if len(numeric_codes) == 0:
        return np.nan

    return max(numeric_codes)


def map_via_unique(series, func):
    """Apply a scalar function once per unique value instead of once per row."""
    uniques = series.dropna().unique()
    mapping = {value: func(value) for value in uniques}
    result = series.map(mapping)

    na_fill = func(np.nan)

    if not pd.isna(na_fill):
        result = result.fillna(na_fill)

    return result


def safe_numeric(series, dtype="float32"):
    return pd.to_numeric(series, errors="coerce").astype(dtype)

In [6]:
def build_positional_read_plan(file_path, delimiter, skiprows=None):
    preview = pd.read_csv(
        file_path,
        sep=delimiter,
        header=None,
        nrows=1,
        skiprows=skiprows,
        dtype="string",
        engine="python"
    )

    names = build_column_names(preview.shape[1])
    usecols = [i for i, name in enumerate(names) if name in SELECTED_RAW_FIELDS]

    return {
        "file_path": file_path,
        "delimiter": delimiter,
        "has_header": False,
        "usecols": usecols,
        "names": names,
        "rename_map": {},
        "selected_count": len(usecols),
        "raw_column_count": len(names),
        "skiprows": skiprows
    }


def get_read_plan(file_path):
    file_path = Path(file_path)
    delimiter = detect_delimiter(file_path)

    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        first_line = f.readline()

    has_header = looks_like_header(first_line, delimiter)

    if has_header:
        header_preview = pd.read_csv(file_path, sep=delimiter, nrows=0, engine="python")
        original_columns = header_preview.columns.tolist()
        standardized_columns = [standardize_column_name(col) for col in original_columns]
        selected_original_columns = [
            original for original, standard in zip(original_columns, standardized_columns)
            if standard in SELECTED_RAW_FIELDS
        ]
        rename_map = {
            original: standard for original, standard in zip(original_columns, standardized_columns)
            if standard in SELECTED_RAW_FIELDS
        }
        plan = {
            "file_path": file_path,
            "delimiter": delimiter,
            "has_header": True,
            "usecols": selected_original_columns,
            "names": None,
            "rename_map": rename_map,
            "selected_count": len(selected_original_columns),
            "raw_column_count": len(original_columns),
            "skiprows": None
        }

        if plan["selected_count"] > 0:
            return plan

        plan = build_positional_read_plan(file_path, delimiter, skiprows=1)
    else:
        plan = build_positional_read_plan(file_path, delimiter)

    if plan["selected_count"] == 0:
        raise ValueError(f"No selected columns were found for {file_path.name}. Detected delimiter: {repr(delimiter)}.")

    return plan


def standardize_loaded_chunk(df, source_file, rename_map=None):
    df = df.copy()

    if rename_map:
        df = df.rename(columns=rename_map)

    df.columns = [standardize_column_name(col) for col in df.columns]
    df = df.loc[:, ~df.columns.duplicated()].copy()
    df["source_file"] = source_file
    return df


def read_fannie_file_chunked(file_path, row_limit=None, chunksize=500_000, sample_fraction=None):
    """Stream a raw file with the C engine, fully processing each chunk before reading the next."""
    plan = get_read_plan(file_path)
    file_path = plan["file_path"]
    chunksize = chunksize or 500_000

    read_kwargs = {
        "filepath_or_buffer": file_path,
        "sep": plan["delimiter"],
        "usecols": plan["usecols"],
        "dtype": str,
        "nrows": row_limit,
        "chunksize": chunksize
    }

    if plan["skiprows"] is not None:
        read_kwargs["skiprows"] = plan["skiprows"]

    if plan["has_header"]:
        read_kwargs["header"] = 0
    else:
        read_kwargs["header"] = None
        read_kwargs["names"] = plan["names"]

    frames = []
    raw_rows = 0
    kept_rows = 0

    with pd.read_csv(**read_kwargs) as reader:
        for chunk in reader:
            raw_rows += len(chunk)
            processed = process_chunk(chunk, file_path.name, plan["rename_map"], sample_fraction)
            kept_rows += len(processed)
            frames.append(processed)
            print(f"  raw rows read: {raw_rows:,} | sampled rows kept: {kept_rows:,}", end="\r")

    print()
    df_file = pd.concat(frames, ignore_index=True)
    return df_file, plan

In [7]:
def clean_missing_codes(df):
    missing_codes = ["", " ", "NA", "N/A", "NULL", "None", "none", "nan", "NaN"]

    for col in df.columns:
        if is_string_dtype(df[col]) or df[col].dtype == "object":
            df[col] = df[col].replace(missing_codes, np.nan)

    return df


def convert_types(df):
    date_cols = [
        "monthly_reporting_period",
        "origination_date",
        "first_payment_date",
        "maturity_date",
        "zero_balance_effective_date",
        "repurchase_date",
        "last_paid_installment_date",
        "foreclosure_date",
        "disposition_date"
    ]

    numeric_cols = [
        "original_interest_rate",
        "current_interest_rate",
        "original_upb",
        "upb_at_issuance",
        "current_actual_upb",
        "original_loan_term",
        "loan_age",
        "remaining_months_to_legal_maturity",
        "remaining_months_to_maturity",
        "original_ltv",
        "original_cltv",
        "number_of_borrowers",
        "dti",
        "borrower_credit_score_at_origination",
        "co_borrower_credit_score_at_origination",
        "number_of_units",
        "mortgage_insurance_percentage",
        "upb_at_time_of_removal",
        "scheduled_principal_current",
        "total_principal_current",
        "unscheduled_principal_current"
    ]

    for col in date_cols:
        if col in df.columns:
            df[col] = parse_mmyyyy_series(df[col])

    for col in numeric_cols:
        if col in df.columns:
            df[col] = safe_numeric(df[col])

    if "current_loan_delinquency_status" in df.columns:
        df["current_dq_num"] = delinquency_to_numeric_series(df["current_loan_delinquency_status"])

    return df


def standardize_string_fields(df):
    string_cols = [
        "loan_identifier",
        "channel",
        "seller_name",
        "servicer_name",
        "first_time_home_buyer_indicator",
        "loan_purpose",
        "property_type",
        "occupancy_status",
        "property_state",
        "msa",
        "zip_code_short",
        "amortization_type",
        "prepayment_penalty_indicator",
        "interest_only_loan_indicator",
        "modification_flag",
        "zero_balance_code",
        "source_file"
    ]

    for col in string_cols:
        if col in df.columns:
            df[col] = df[col].astype("string").str.strip()

    if "zip_code_short" in df.columns:
        df["zip_code_short"] = df["zip_code_short"].str.replace(r"\.0$", "", regex=True)
        df["zip_code_short"] = df["zip_code_short"].str.zfill(3)

    return df


def filter_sampled_loans(df, sample_fraction):
    if sample_fraction is None or sample_fraction >= 1 or "loan_identifier" not in df.columns:
        return df

    # Deterministic hash so the same loans are kept in every chunk, file, and rerun.
    loan_hash = pd.util.hash_pandas_object(df["loan_identifier"].astype(str), index=False)
    keep_mask = (loan_hash % 10_000) < int(round(sample_fraction * 10_000))

    return df.loc[keep_mask.to_numpy()]


def extract_payment_history_features(df, lookback=12):
    if "loan_payment_history" not in df.columns:
        return df

    history = df["loan_payment_history"].astype("string")

    df[f"count_30plus_dq_past_{lookback}m"] = safe_numeric(
        map_via_unique(history, lambda x: payment_history_count_dq(x, threshold=1, lookback=lookback))
    )
    df[f"count_60plus_dq_past_{lookback}m"] = safe_numeric(
        map_via_unique(history, lambda x: payment_history_count_dq(x, threshold=2, lookback=lookback))
    )
    df[f"max_dq_past_{lookback}m"] = safe_numeric(
        map_via_unique(history, lambda x: payment_history_max_dq(x, lookback=lookback))
    )

    # The raw history string is the widest text field; drop it once its features exist.
    return df.drop(columns=["loan_payment_history"])


def process_chunk(chunk, source_file, rename_map=None, sample_fraction=None):
    chunk = standardize_loaded_chunk(chunk, source_file, rename_map)
    chunk = filter_sampled_loans(chunk, sample_fraction)
    chunk = clean_missing_codes(chunk)
    chunk = convert_types(chunk)
    chunk = standardize_string_fields(chunk)
    chunk = extract_payment_history_features(chunk)
    return chunk


CATEGORY_COLS = [
    "channel",
    "seller_name",
    "servicer_name",
    "first_time_home_buyer_indicator",
    "loan_purpose",
    "property_type",
    "occupancy_status",
    "property_state",
    "msa",
    "zip_code_short",
    "amortization_type",
    "prepayment_penalty_indicator",
    "interest_only_loan_indicator",
    "modification_flag",
    "zero_balance_code",
    "current_loan_delinquency_status",
    "source_file"
]


def convert_to_category(df, columns=None):
    columns = CATEGORY_COLS if columns is None else columns

    for col in columns:
        if col in df.columns:
            df[col] = df[col].astype("category")

    return df

In [8]:
def create_behavior_features(df):
    df = df.copy()

    if "current_dq_num" in df.columns:
        df["is_current"] = (df["current_dq_num"] == 0).astype("Int64")
        df["is_30plus_dq"] = (df["current_dq_num"] >= 1).astype("Int64")
        df["is_60plus_dq"] = (df["current_dq_num"] >= 2).astype("Int64")

    if {"loan_identifier", "monthly_reporting_period", "current_dq_num"}.issubset(df.columns):
        df = df.sort_values(["loan_identifier", "monthly_reporting_period"])
        df["prior_dq_num"] = df.groupby("loan_identifier")["current_dq_num"].shift(1)
        df["dq_status_change"] = df["current_dq_num"] - df["prior_dq_num"]
        df["dq_status_worsened"] = (df["dq_status_change"] > 0).astype("Int64")

    if {"current_actual_upb", "original_upb"}.issubset(df.columns):
        df["upb_ratio"] = df["current_actual_upb"] / df["original_upb"]
        df["paydown_pct"] = 1 - df["upb_ratio"]

    if "loan_payment_history" in df.columns:
        df["count_30plus_dq_past_12m"] = df["loan_payment_history"].map(
            lambda x: payment_history_count_dq(x, threshold=1, lookback=12)
        )
        df["count_60plus_dq_past_12m"] = df["loan_payment_history"].map(
            lambda x: payment_history_count_dq(x, threshold=2, lookback=12)
        )
        df["max_dq_past_12m"] = df["loan_payment_history"].map(
            lambda x: payment_history_max_dq(x, lookback=12)
        )

    return df


def create_future_serious_dq_target(df, horizon_months=6, serious_threshold=3):
    required_cols = {"loan_identifier", "monthly_reporting_period", "current_dq_num"}
    missing_required = required_cols - set(df.columns)

    if missing_required:
        raise KeyError(f"Missing required columns for target creation: {missing_required}")

    df = df.copy()
    df = df.sort_values(["loan_identifier", "monthly_reporting_period"]).reset_index(drop=True)

    lead_cols = []

    for month_ahead in range(1, horizon_months + 1):
        lead_col = f"dq_lead_{month_ahead}m"
        df[lead_col] = df.groupby("loan_identifier")["current_dq_num"].shift(-month_ahead)
        lead_cols.append(lead_col)

    future_max_dq = df[lead_cols].max(axis=1, skipna=True)
    has_future_observation = df[lead_cols].notna().any(axis=1)
    target_col = f"future_serious_dq_{horizon_months}m"

    df[target_col] = np.where(
        has_future_observation,
        (future_max_dq >= serious_threshold).astype(int),
        np.nan
    )

    already_serious = df["current_dq_num"] >= serious_threshold
    df.loc[already_serious, target_col] = np.nan
    df = df.drop(columns=lead_cols)

    return df

## 4. Find the raw files

This section finds the raw Fannie Mae files in `data/raw/` and filters them using the quarter tokens set in the setup section.

In [9]:
raw_patterns = [
    str(RAW_DIR / "*.csv"),
    str(RAW_DIR / "*.txt"),
    str(RAW_DIR / "*.dat")
]

raw_files = []

for pattern in raw_patterns:
    raw_files.extend(glob.glob(pattern))

raw_files = sorted(raw_files)

if RAW_FILE_TOKENS:
    raw_files = [
        file for file in raw_files
        if any(token.lower() in Path(file).name.lower() for token in RAW_FILE_TOKENS)
    ]

if MAX_FILES is not None:
    raw_files = raw_files[:MAX_FILES]

print("Raw files found:", len(raw_files))

for file in raw_files:
    print(file)

if len(raw_files) == 0:
    raise FileNotFoundError(
        f"No matching raw files found in {RAW_DIR}. Update RAW_FILE_TOKENS or place the raw files in data/raw/."
    )

Raw files found: 2
C:\Users\Esh\OneDrive\Documents\Mortgage Delinquency Forecasting Risk\data\raw\2021Q1.csv
C:\Users\Esh\OneDrive\Documents\Mortgage Delinquency Forecasting Risk\data\raw\2021Q2.csv


## 5. Preview the read plan

Before loading the data, I check how many columns each raw file has and how many selected columns will be loaded.

In [10]:
read_plans = []

for file_path in raw_files:
    plan = get_read_plan(file_path)
    read_plans.append(plan)

read_plan_summary = pd.DataFrame({
    "file": [plan["file_path"].name for plan in read_plans],
    "delimiter": [plan["delimiter"] for plan in read_plans],
    "has_header": [plan["has_header"] for plan in read_plans],
    "raw_column_count": [plan["raw_column_count"] for plan in read_plans],
    "selected_column_count": [plan["selected_count"] for plan in read_plans]
})

read_plan_summary["columns_skipped"] = (
    read_plan_summary["raw_column_count"] - read_plan_summary["selected_column_count"]
)

read_plan_summary.to_csv(RAW_PROFILE_OUTPUT, index=False)
display(read_plan_summary)
print("Saved raw profile to:", RAW_PROFILE_OUTPUT)

,file,delimiter,has_header,raw_column_count,selected_column_count,columns_skipped
0,2021Q1.csv,|,False,113,49,64
1,2021Q2.csv,|,False,113,49,64


Saved raw profile to: C:\Users\Esh\OneDrive\Documents\Mortgage Delinquency Forecasting Risk\data\processed\raw_profile_2021q1_q2_sample10pct.csv


## 6. Process each quarter in chunks

Each raw file is streamed with the pandas C engine in chunks of `CHUNKSIZE` rows. Every chunk is standardized, sampled by loan, cleaned, and type-converted before the next chunk is read, so memory stays bounded. After a file finishes loading, duplicates are removed and the behavior features and the 6-month target are built for that quarter on its own. Loans never span quarter files, so per-quarter labeling is safe; this is verified after combining.

In [11]:
if LOAN_SAMPLE_FRACTION is None or LOAN_SAMPLE_FRACTION >= 1:
    print("Loan sampling: OFF (processing every loan)")
else:
    print(f"Loan sampling: keeping ~{LOAN_SAMPLE_FRACTION:.0%} of loans")

file_panels = []

for file_path in raw_files:
    file_name = Path(file_path).name
    print(f"\n=== Processing {file_name} ===")

    df_file, plan = read_fannie_file_chunked(
        file_path,
        row_limit=ROW_LIMIT,
        chunksize=CHUNKSIZE,
        sample_fraction=LOAN_SAMPLE_FRACTION
    )

    print("Loaded shape:", df_file.shape)

    before_rows = len(df_file)
    df_file = df_file.drop_duplicates()
    df_file = (
        df_file.sort_values(["loan_identifier", "monthly_reporting_period", "source_file"])
        .drop_duplicates(subset=["loan_identifier", "monthly_reporting_period"], keep="last")
        .reset_index(drop=True)
    )
    print("Duplicate rows removed:", f"{before_rows - len(df_file):,}")

    df_file = create_behavior_features(df_file)
    df_file = create_future_serious_dq_target(
        df_file,
        horizon_months=HORIZON_MONTHS,
        serious_threshold=SERIOUS_DQ_THRESHOLD
    )

    print("Quarter panel shape:", df_file.shape)
    print("Memory usage MB:", round(df_file.memory_usage(deep=True).sum() / 1_000_000, 2))

    file_panels.append(df_file)
    del df_file
    gc.collect()

Loan sampling: keeping ~10% of loans

=== Processing 2021Q1.csv ===


  raw rows read: 73,467,360 | sampled rows kept: 7,339,519


Loaded shape: (7339519, 53)


Duplicate rows removed: 0


Quarter panel shape: (7339519, 62)


...

  raw rows read: 65,794,514 | sampled rows kept: 6,576,182


Loaded shape: (6576182, 53)


Duplicate rows removed: 0


Quarter panel shape: (6576182, 62)


Memory usage MB: 7943.36


## 7. Combine quarters

The per-quarter panels are concatenated into one loan-month panel. Because the behavior features and the target were built per file, this section verifies that no loan appears in more than one raw file, then converts low-cardinality text fields to categories to reduce memory.

In [12]:
loan_id_sets = [set(panel["loan_identifier"].dropna().unique()) for panel in file_panels]

overlap = set()

for i in range(len(loan_id_sets)):
    for j in range(i + 1, len(loan_id_sets)):
        overlap |= loan_id_sets[i] & loan_id_sets[j]

if overlap:
    print(
        f"WARNING: {len(overlap):,} loans appear in more than one raw file. "
        "Behavior features and the target were built per file, so these loans' "
        "timelines are split across files. Review before modeling."
    )
else:
    print("No loans appear in more than one raw file. Per-quarter target creation is safe.")

df = pd.concat(file_panels, ignore_index=True)

del file_panels, loan_id_sets
gc.collect()

df = convert_to_category(df)

print("Combined panel shape:", df.shape)
print("Memory usage MB:", round(df.memory_usage(deep=True).sum() / 1_000_000, 2))
display(df.head())

No loans appear in more than one raw file. Per-quarter target creation is safe.


Combined panel shape: (13915701, 62)
Memory usage MB: 4453.16


,reference_pool_id,loan_identifier,monthly_reporting_period,channel,seller_name,servicer_name,original_interest_rate,current_interest_rate,original_upb,upb_at_issuance,current_actual_upb,original_loan_term,origination_date,first_payment_date,loan_age,remaining_months_to_legal_maturity,remaining_months_to_maturity,maturity_date,original_ltv,original_cltv,number_of_borrowers,dti,borrower_credit_score_at_origination,co_borrower_credit_score_at_origination,first_time_home_buyer_indicator,loan_purpose,property_type,number_of_units,occupancy_status,property_state,msa,zip_code_short,mortgage_insurance_percentage,amortization_type,prepayment_penalty_indicator,interest_only_loan_indicator,current_loan_delinquency_status,modification_flag,zero_balance_code,zero_balance_effective_date,upb_at_time_of_removal,repurchase_date,scheduled_principal_current,total_principal_current,unscheduled_principal_current,last_paid_installment_date,foreclosure_date,disposition_date,source_file,current_dq_num,count_30plus_dq_past_12m,count_60plus_dq_past_12m,max_dq_past_12m,is_current,is_30plus_dq,is_60plus_dq,prior_dq_num,dq_status_change,dq_status_worsened,upb_ratio,paydown_pct,future_serious_dq_6m
0,NaN,000122128334,2021-01-01,R,"loanDepot.com, LLC","loanDepot.com, LLC",2.3750,2.3750,"766,000.0000",NaN,"766,000.0000",360.0000,2020-12-01,2021-02-01,0.0000,360.0000,360.0000,2051-01-01,54.0000,54.0000,2.0000,23.0000,761.0000,760.0000,N,R,PU,1.0000,P,CA,41940,940,NaN,FRM,N,N,00,N,<NA>,NaT,NaN,NaT,NaN,NaN,NaN,NaT,NaT,NaT,2021Q1.csv,0.0000,0.0000,0.0000,0.0000,1,0,0,NaN,NaN,0,1.0000,0.0000,0.0000
1,NaN,000122128334,2021-02-01,R,"loanDepot.com, LLC","loanDepot.com, LLC",2.3750,2.3750,"766,000.0000",NaN,"764,000.0000",360.0000,2020-12-01,2021-02-01,1.0000,359.0000,359.0000,2051-01-01,54.0000,54.0000,2.0000,23.0000,761.0000,760.0000,N,R,PU,1.0000,P,CA,41940,940,NaN,FRM,N,N,00,N,<NA>,NaT,NaN,NaT,NaN,"1,484.7500",NaN,NaT,NaT,NaT,2021Q1.csv,0.0000,0.0000,0.0000,0.0000,1,0,0,0.0000,0.0000,0,0.9974,0.0026,0.0000
2,NaN,000122128334,2021-03-01,R,"loanDepot.com, LLC","loanDepot.com, LLC",2.3750,2.3750,"766,000.0000",NaN,"763,000.0000",360.0000,2020-12-01,2021-02-01,2.0000,358.0000,358.0000,2051-01-01,54.0000,54.0000,2.0000,23.0000,761.0000,760.0000,N,R,PU,1.0000,P,CA,41940,940,NaN,FRM,N,N,00,N,<NA>,NaT,NaN,NaT,NaN,"1,487.6899",NaN,NaT,NaT,NaT,2021Q1.csv,0.0000,0.0000,0.0000,0.0000,1,0,0,0.0000,0.0000,0,0.9961,0.0039,0.0000
3,NaN,000122128334,2021-04-01,R,"loanDepot.com, LLC","loanDepot.com, LLC",2.3750,2.3750,"766,000.0000",NaN,"761,000.0000",360.0000,2020-12-01,2021-02-01,3.0000,357.0000,357.0000,2051-01-01,54.0000,54.0000,2.0000,23.0000,761.0000,760.0000,N,R,PU,1.0000,P,CA,41940,940,NaN,FRM,N,N,00,N,<NA>,NaT,NaN,NaT,NaN,"1,490.6300",NaN,NaT,NaT,NaT,2021Q1.csv,0.0000,0.0000,0.0000,0.0000,1,0,0,0.0000,0.0000,0,0.9935,0.0065,0.0000
4,NaN,000122128334,2021-05-01,R,"loanDepot.com, LLC","loanDepot.com, LLC",2.3750,2.3750,"766,000.0000",NaN,"760,000.0000",360.0000,2020-12-01,2021-02-01,4.0000,356.0000,356.0000,2051-01-01,54.0000,54.0000,2.0000,23.0000,761.0000,760.0000,N,R,PU,1.0000,P,CA,41940,940,NaN,FRM,N,N,00,N,<NA>,NaT,NaN,NaT,NaN,"1,493.5800",NaN,NaT,NaT,NaT,2021Q1.csv,0.0000,0.0000,0.0000,0.0000,1,0,0,0.0000,0.0000,0,0.9922,0.0078,0.0000


In [13]:
required_columns = [
    "loan_identifier",
    "monthly_reporting_period",
    "current_loan_delinquency_status"
]

missing_required = [col for col in required_columns if col not in df.columns]

if missing_required:
    raise KeyError(
        f"Missing required columns after loading: {missing_required}. Check the field layout or selected raw fields."
    )

print("Required columns are present.")

Required columns are present.


In [14]:
duplicate_loan_months = df.duplicated(subset=["loan_identifier", "monthly_reporting_period"]).sum()

print("Duplicate loan-month rows in combined panel:", f"{duplicate_loan_months:,}")

if duplicate_loan_months > 0:
    df = (
        df.sort_values(["loan_identifier", "monthly_reporting_period", "source_file"])
        .drop_duplicates(subset=["loan_identifier", "monthly_reporting_period"], keep="last")
        .reset_index(drop=True)
    )
    print(
        "WARNING: cross-file duplicate loan-months were removed, but behavior features and "
        "the target were computed per file. Rebuild with combined labeling if this persists."
    )
    print("Shape after cleanup:", df.shape)

Duplicate loan-month rows in combined panel: 0


In [15]:
df = df.sort_values(["loan_identifier", "monthly_reporting_period"]).reset_index(drop=True)

df["reporting_year"] = df["monthly_reporting_period"].dt.year
df["reporting_month"] = df["monthly_reporting_period"].dt.month
df["reporting_quarter"] = df["monthly_reporting_period"].dt.to_period("Q").astype(str)

if "origination_date" in df.columns:
    df["origination_year"] = df["origination_date"].dt.year
    df["origination_quarter"] = df["origination_date"].dt.to_period("Q").astype(str)

df["loan_month_key"] = (
    df["loan_identifier"].astype(str)
    + "_"
    + df["monthly_reporting_period"].dt.strftime("%Y_%m")
)

display(df[["loan_identifier", "monthly_reporting_period", "loan_month_key"]].head())

,loan_identifier,monthly_reporting_period,loan_month_key
0,000122128334,2021-01-01,000122128334_2021_01
1,000122128334,2021-02-01,000122128334_2021_02
2,000122128334,2021-03-01,000122128334_2021_03
3,000122128334,2021-04-01,000122128334_2021_04
4,000122128334,2021-05-01,000122128334_2021_05


## 8. Timeline and quality checks

Since the target depends on future monthly observations, I check the loan timelines, reporting-period coverage, missingness, and delinquency-status distribution.

In [16]:
loan_timeline = (
    df.groupby("loan_identifier")
    .agg(
        first_month=("monthly_reporting_period", "min"),
        last_month=("monthly_reporting_period", "max"),
        observed_months=("monthly_reporting_period", "nunique")
    )
    .reset_index()
)

loan_timeline["first_month_index"] = (
    loan_timeline["first_month"].dt.year * 12
    + loan_timeline["first_month"].dt.month
)

loan_timeline["last_month_index"] = (
    loan_timeline["last_month"].dt.year * 12
    + loan_timeline["last_month"].dt.month
)

loan_timeline["expected_months"] = (
    loan_timeline["last_month_index"]
    - loan_timeline["first_month_index"]
    + 1
)

loan_timeline["missing_months_inside_timeline"] = (
    loan_timeline["expected_months"]
    - loan_timeline["observed_months"]
)

display(loan_timeline["missing_months_inside_timeline"].describe())
display(loan_timeline.sort_values("missing_months_inside_timeline", ascending=False).head(20))

count   272,990.0000
mean          0.0000
std           0.0000
min           0.0000
25%           0.0000
50%           0.0000
75%           0.0000
max           0.0000
Name: missing_months_inside_timeline, dtype: float64

,loan_identifier,first_month,last_month,observed_months,first_month_index,last_month_index,expected_months,missing_months_inside_timeline
0,000122128334,2021-01-01,2025-12-01,60,24253,24312,60,0
181998,000124095879,2021-04-01,2025-12-01,57,24256,24312,57,0
181984,000124095765,2021-04-01,2025-12-01,57,24256,24312,57,0
181985,000124095777,2021-04-01,2025-12-01,57,24256,24312,57,0
181986,000124095786,2021-04-01,2025-07-01,52,24256,24307,52,0
181987,000124095789,2021-04-01,2025-12-01,57,24256,24312,57,0
181988,000124095791,2021-04-01,2025-12-01,57,24256,24312,57,0
181989,000124095795,2021-04-01,2025-12-01,57,24256,24312,57,0
181990,000124095818,2021-04-01,2023-04-01,25,24256,24280,25,0
181991,000124095819,2021-04-01,2025-12-01,57,24256,24312,57,0


In [17]:
missing_summary = (
    df.isna()
    .mean()
    .mul(100)
    .rename("missing_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
    .sort_values("missing_pct", ascending=False)
)

display(missing_summary.head(50))

,feature,missing_pct
0,reference_pool_id,100.0000
42,scheduled_principal_current,100.0000
41,repurchase_date,100.0000
9,upb_at_issuance,100.0000
44,unscheduled_principal_current,100.0000
47,disposition_date,99.9994
45,last_paid_installment_date,99.9992
46,foreclosure_date,99.9992
40,upb_at_time_of_removal,99.5860
39,zero_balance_effective_date,99.5860


In [18]:
coverage_summary = pd.DataFrame({
    "metric": [
        "rows",
        "unique_loans",
        "unique_reporting_periods",
        "min_reporting_period",
        "max_reporting_period",
        "duplicate_loan_months_after_cleaning"
    ],
    "value": [
        len(df),
        df["loan_identifier"].nunique(),
        df["monthly_reporting_period"].nunique(),
        df["monthly_reporting_period"].min(),
        df["monthly_reporting_period"].max(),
        df.duplicated(subset=["loan_identifier", "monthly_reporting_period"]).sum()
    ]
})

display(coverage_summary)

,metric,value
0,rows,13915701
1,unique_loans,272990
2,unique_reporting_periods,60
3,min_reporting_period,2021-01-01 00:00:00
4,max_reporting_period,2025-12-01 00:00:00
5,duplicate_loan_months_after_cleaning,0


In [19]:
dq_status_summary = (
    df["current_dq_num"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("current_dq_num")
    .reset_index(name="count")
)

dq_status_summary["pct"] = dq_status_summary["count"] / len(df) * 100

display(dq_status_summary)

,current_dq_num,count,pct
0,0.0000,13793673,99.1231
1,1.0000,73346,0.5271
2,2.0000,15360,0.1104
3,3.0000,7158,0.0514
4,4.0000,5102,0.0367
5,5.0000,4302,0.0309
6,6.0000,3104,0.0223
7,7.0000,2406,0.0173
8,8.0000,2025,0.0146
9,9.0000,1691,0.0122


## 9. Review monthly behavior features

The behavior features were created during per-quarter processing. They use information available as of the current reporting month and are used later in preprocessing and modeling.

In [20]:
behavior_cols = [
    "current_dq_num",
    "prior_dq_num",
    "dq_status_change",
    "dq_status_worsened",
    "upb_ratio",
    "paydown_pct",
    "count_30plus_dq_past_12m",
    "count_60plus_dq_past_12m",
    "max_dq_past_12m"
]

available_behavior_cols = [col for col in behavior_cols if col in df.columns]

display(df[["loan_identifier", "monthly_reporting_period"] + available_behavior_cols].head(20))

,loan_identifier,monthly_reporting_period,current_dq_num,prior_dq_num,dq_status_change,dq_status_worsened,upb_ratio,paydown_pct,count_30plus_dq_past_12m,count_60plus_dq_past_12m,max_dq_past_12m
0,000122128334,2021-01-01,0.0000,NaN,NaN,0,1.0000,0.0000,0.0000,0.0000,0.0000
1,000122128334,2021-02-01,0.0000,0.0000,0.0000,0,0.9974,0.0026,0.0000,0.0000,0.0000
2,000122128334,2021-03-01,0.0000,0.0000,0.0000,0,0.9961,0.0039,0.0000,0.0000,0.0000
3,000122128334,2021-04-01,0.0000,0.0000,0.0000,0,0.9935,0.0065,0.0000,0.0000,0.0000
4,000122128334,2021-05-01,0.0000,0.0000,0.0000,0,0.9922,0.0078,0.0000,0.0000,0.0000
5,000122128334,2021-06-01,0.0000,0.0000,0.0000,0,0.9896,0.0104,0.0000,0.0000,0.0000
6,000122128334,2021-07-01,0.0000,0.0000,0.0000,0,0.9883,0.0117,0.0000,0.0000,0.0000
7,000122128334,2021-08-01,0.0000,0.0000,0.0000,0,0.9858,0.0142,0.0000,0.0000,0.0000
8,000122128334,2021-09-01,0.0000,0.0000,0.0000,0,0.9839,0.0161,0.0000,0.0000,0.0000
9,000122128334,2021-10-01,0.0000,0.0000,0.0000,0,0.9623,0.0377,0.0000,0.0000,0.0000


## 10. Review future delinquency target

The target was created during per-quarter processing. It is positive when a loan becomes 90 or more days delinquent within the next 6 observed reporting months. Rows without a future observation window are left unlabeled.

In [21]:
target_summary = (
    df[TARGET_COL]
    .value_counts(dropna=False)
    .rename_axis(TARGET_COL)
    .reset_index(name="count")
)

target_summary["pct"] = target_summary["count"] / len(df) * 100

display(target_summary)

target_summary.to_csv(TARGET_SUMMARY_OUTPUT, index=False)
print("Saved target summary to:", TARGET_SUMMARY_OUTPUT)

,future_serious_dq_6m,count,pct
0,0.0000,13577132,97.5670
1,NaN,305482,2.1952
2,1.0000,33087,0.2378


Saved target summary to: C:\Users\Esh\OneDrive\Documents\Mortgage Delinquency Forecasting Risk\data\processed\target_summary_2021q1_q2_sample10pct.csv


In [22]:
labeled_df = df.dropna(subset=[TARGET_COL])

if len(labeled_df) > 0:
    sample_loans = (
        labeled_df
        .sample(min(3, labeled_df["loan_identifier"].nunique()), random_state=42)["loan_identifier"]
        .unique()
    )

    timeline_cols = [
        "loan_identifier",
        "monthly_reporting_period",
        "current_loan_delinquency_status",
        "current_dq_num",
        TARGET_COL
    ]

    extra_cols = [
        "loan_age",
        "current_actual_upb",
        "upb_ratio",
        "count_30plus_dq_past_12m",
        "max_dq_past_12m",
        "zero_balance_code"
    ]

    timeline_cols += [col for col in extra_cols if col in df.columns]

    display(
        df[df["loan_identifier"].isin(sample_loans)]
        .sort_values(["loan_identifier", "monthly_reporting_period"])
        [timeline_cols]
        .head(100)
    )
else:
    print("No labeled rows were available for timeline sampling.")

,loan_identifier,monthly_reporting_period,current_loan_delinquency_status,current_dq_num,future_serious_dq_6m,loan_age,current_actual_upb,upb_ratio,count_30plus_dq_past_12m,max_dq_past_12m,zero_balance_code
1145045,000122349108,2021-01-01,00,0.0000,0.0000,0.0000,"225,000.0000",1.0000,0.0000,0.0000,<NA>
1145046,000122349108,2021-02-01,00,0.0000,0.0000,1.0000,"225,000.0000",1.0000,0.0000,0.0000,<NA>
1145047,000122349108,2021-03-01,00,0.0000,0.0000,2.0000,"224,000.0000",0.9956,0.0000,0.0000,<NA>
1145048,000122349108,2021-04-01,00,0.0000,0.0000,3.0000,"224,000.0000",0.9956,0.0000,0.0000,<NA>
1145049,000122349108,2021-05-01,00,0.0000,0.0000,4.0000,"223,000.0000",0.9911,0.0000,0.0000,<NA>
1145050,000122349108,2021-06-01,00,0.0000,0.0000,5.0000,"223,000.0000",0.9911,0.0000,0.0000,<NA>
1145051,000122349108,2021-07-01,00,0.0000,0.0000,6.0000,"222,000.0000",0.9867,0.0000,0.0000,<NA>
1145052,000122349108,2021-08-01,00,0.0000,0.0000,7.0000,"221,629.8906",0.9850,0.0000,0.0000,<NA>
1145053,000122349108,2021-09-01,00,0.0000,0.0000,8.0000,"221,144.0156",0.9829,0.0000,0.0000,<NA>
1145054,000122349108,2021-10-01,00,0.0000,0.0000,9.0000,"220,657.0469",0.9807,0.0000,0.0000,<NA>


## 11. Leakage review

These fields are kept for review, but they should not be used as standard predictors because they may describe events after the prediction point.

In [23]:
leakage_candidates = [
    "zero_balance_code",
    "zero_balance_effective_date",
    "upb_at_time_of_removal",
    "repurchase_date",
    "foreclosure_date",
    "disposition_date"
]

leakage_fields_present = [col for col in leakage_candidates if col in df.columns]

leakage_review = pd.DataFrame({
    "feature": leakage_fields_present,
    "recommended_modeling_use": "exclude_from_predictors",
    "reason": "Event, removal, foreclosure, or disposition information can leak future outcome information."
})

display(leakage_review)

,feature,recommended_modeling_use,reason
0,zero_balance_code,exclude_from_predictors,"Event, removal, foreclosure, or disposition in..."
1,zero_balance_effective_date,exclude_from_predictors,"Event, removal, foreclosure, or disposition in..."
2,upb_at_time_of_removal,exclude_from_predictors,"Event, removal, foreclosure, or disposition in..."
3,repurchase_date,exclude_from_predictors,"Event, removal, foreclosure, or disposition in..."
4,foreclosure_date,exclude_from_predictors,"Event, removal, foreclosure, or disposition in..."
5,disposition_date,exclude_from_predictors,"Event, removal, foreclosure, or disposition in..."


## 12. Save cleaned outputs

This section saves the tagged processed loan-month panel, data dictionary, target summary, and timeline gap check.

In [24]:
data_dictionary = pd.DataFrame({
    "feature": df.columns,
    "dtype": [str(df[col].dtype) for col in df.columns],
    "missing_pct": [df[col].isna().mean() * 100 for col in df.columns],
    "n_unique": [df[col].nunique(dropna=True) for col in df.columns]
}).sort_values("feature")

data_dictionary.to_csv(DATA_DICTIONARY_OUTPUT, index=False)
print("Saved data dictionary to:", DATA_DICTIONARY_OUTPUT)

display(data_dictionary.head(50))

Saved data dictionary to: C:\Users\Esh\OneDrive\Documents\Mortgage Delinquency Forecasting Risk\data\processed\data_dictionary_2021q1_q2_sample10pct.csv


,feature,dtype,missing_pct,n_unique
33,amortization_type,category,0.0000,1
22,borrower_credit_score_at_origination,float32,0.0368,216
3,channel,category,0.0000,3
23,co_borrower_credit_score_at_origination,float32,52.6960,216
50,count_30plus_dq_past_12m,float32,0.0000,13
51,count_60plus_dq_past_12m,float32,0.0000,13
10,current_actual_upb,float32,0.0000,7627517
49,current_dq_num,float32,0.0000,43
7,current_interest_rate,float32,0.4132,1362
36,current_loan_delinquency_status,category,0.0000,43


In [25]:
try:
    df.to_parquet(PANEL_OUTPUT_PARQUET, index=False)
    print("Saved processed panel to:", PANEL_OUTPUT_PARQUET)
except Exception as e:
    print("Could not save parquet. Saving CSV instead.")
    print("Parquet error:", e)
    df.to_csv(PANEL_OUTPUT_CSV, index=False)
    print("Saved processed panel to:", PANEL_OUTPUT_CSV)

Saved processed panel to: C:\Users\Esh\OneDrive\Documents\Mortgage Delinquency Forecasting Risk\data\processed\loan_month_panel_2021q1_q2_sample10pct.parquet


In [26]:
loan_timeline.to_csv(LOAN_TIMELINE_OUTPUT, index=False)

print("Saved loan timeline gap check to:")
print(LOAN_TIMELINE_OUTPUT)

Saved loan timeline gap check to:
C:\Users\Esh\OneDrive\Documents\Mortgage Delinquency Forecasting Risk\data\processed\loan_timeline_gap_check_2021q1_q2_sample10pct.csv


In [27]:
final_summary = pd.DataFrame({
    "metric": [
        "dataset_tag",
        "loan_sample_fraction",
        "processed_rows",
        "processed_columns",
        "unique_loans",
        "unique_reporting_periods",
        "target_positive_rate",
        "target_valid_rows",
        "output_parquet_exists",
        "output_csv_exists"
    ],
    "value": [
        DATASET_TAG,
        LOAN_SAMPLE_FRACTION,
        len(df),
        df.shape[1],
        df["loan_identifier"].nunique(),
        df["monthly_reporting_period"].nunique(),
        df[TARGET_COL].mean(skipna=True),
        df[TARGET_COL].notna().sum(),
        PANEL_OUTPUT_PARQUET.exists(),
        PANEL_OUTPUT_CSV.exists()
    ]
})

display(final_summary)

,metric,value
0,dataset_tag,2021q1_q2_sample10pct
1,loan_sample_fraction,0.1000
2,processed_rows,13915701
3,processed_columns,68
4,unique_loans,272990
5,unique_reporting_periods,60
6,target_positive_rate,0.0024
7,target_valid_rows,13610219
8,output_parquet_exists,True
9,output_csv_exists,False


## 13. Data wrangling summary

This notebook streams the selected Fannie Mae performance fields from each quarterly file in chunks with the C engine, samples loans deterministically when `LOAN_SAMPLE_FRACTION` is set, standardizes the loan-month data with vectorized conversions, creates current-month behavior features, and builds the 6-month serious delinquency target per quarter before combining.

The output is a tagged processed panel that can be used by the EDA, preprocessing, and modeling notebooks without mixing this run with older sample outputs. Sampled runs are tagged with the sample share (for example `2021q1_q2_sample10pct`), so a later full run will not overwrite them.